# Analiza dialogowa scenariusza: kto mówi, ile i czyim głosem

## Wersja studencka — moduł 2: GŁOS I PŁEĆ

W poprzednim module zbudowałeś *sieć współwystąpień* — pokazała, kto z kim **pojawia się** w scenach. Ale scenariusz wie znacznie więcej: wie, **kto mówi**, **ile mówi** i **czyimi ustami** prowadzona jest narracja. Ten notatnik wydobywa z tego samego scenariusza warstwę **dialogów**.

Efektem końcowym będzie:
1. odzyskanie czystego tekstu scenariusza (jak w module sieciowym),
2. wydzielenie scen i przypisanie do nich kwestii dialogowych,
3. tabela: która postać wypowiada które kwestie i ile słów,
4. otagowanie postaci płcią,
5. wykres udziału w dialogu z podziałem na płeć,
6. zapis plików `dialogi.csv` i `postacie.csv` do dalszej analizy emocji.

Tak jak poprzednio: nie piszesz kodu samodzielnie. Każdy krok to mały prompt, który przekazujesz modelowi AI, a wygenerowany kod wklejasz do pustej komórki pod spodem.

## Jak pracować z tym notatnikiem

1. Zmień adres scenariusza w komórce parametru (najlepiej ten sam film, co w module sieciowym).
2. Skopiuj prompt z bieżącego kroku do modelu AI.
3. Wklej otrzymany kod do pustej komórki pod promptem.
4. Uruchom kod i porównaj rezultat z sekcją **Po uruchomieniu powinieneś zobaczyć**.
5. Jeśli wynik nie zgadza się z opisem, popraw tylko bieżący krok.
6. Dopiero po uzyskaniu poprawnego wyniku przejdź dalej.

Najważniejszy wynik każdego kroku powinien pozostać dostępny do następnego kroku, ale to model ma zdecydować, jak to zorganizować w kodzie.

## Parametr startowy

To jedyna komórka, którą zmieniasz ręcznie. Wybierz adres strony scenariusza z IMSDb — najlepiej ten sam film, dla którego budowałeś sieć, żeby móc porównać oba moduły.

In [ ]:
# Jedyny parametr, który zmieniasz w tym notatniku
adres_scenariusza = "https://imsdb.com/scripts/Pulp-Fiction.html"

---
## Etap 1: Odzyskanie czystego tekstu scenariusza

Zaczynamy od tego samego punktu, co moduł sieciowy: pobieramy stronę i wydobywamy z niej sam tekst scenariusza. Znasz już ten krok, więc robimy go zwięźle — w jednym podejściu.

### Krok 1A. Pobranie strony i wydobycie tekstu scenariusza

#### Cel i sens analityczny

Cała dalsza analiza dialogów opiera się na czystym tekście scenariusza. Musimy najpierw odzyskać go z podanego adresu — bez menu, stopek i elementów witryny.

#### Prompt dla modelu

```text
Kontekst:
Pracujesz na stronie scenariusza filmowego z serwisu IMSDb. Strona zawiera menu, linki i elementy witryny, których nie chcemy.

Wejście:
Adres scenariusza zapisany w komórce parametru.

Zadanie:
Pobierz stronę i wydobądź z niej wyłącznie właściwy tekst scenariusza (dialogi i didaskalia), odrzucając nawigację, nagłówki i stopki witryny. Zachowaj oczyszczony tekst w zmiennej dostępnej w kolejnych krokach.

Pokaż wynik:
- komunikat, czy pobranie i oczyszczenie się udało,
- liczbę znaków i liczbę linii oczyszczonego tekstu,
- pierwsze 15 niepustych linii.

Warunek poprawności:
Podgląd powinien przypominać zapis scenariusza (nagłówki scen, nazwy postaci, kwestie), a nie elementy strony internetowej. Tekst nie może być pusty ani podejrzanie krótki.

Jeśli wystąpi błąd:
Wyjaśnij, czy problem dotyczy adresu, połączenia, czy nieznalezienia głównego bloku z tekstem scenariusza.

Nie rób jeszcze:
Nie wykrywaj scen ani postaci, nie analizuj dialogów.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Potwierdzenie, że tekst scenariusza został pobrany i oczyszczony.
- Liczbę znaków i linii oraz kilkanaście pierwszych linii wyglądających jak scenariusz.
- Brak widocznego menu, stopki i innych elementów witryny.

---
## Etap 2: Od tekstu do kwestii dialogowych

Teraz najważniejszy, nowy krok tego modułu: zamieniamy ciągły tekst w **tabelę kwestii** — kto, w której scenie, co powiedział. To będzie surowiec do liczenia mowy i emocji.

### Krok 2A. Wykrycie granic scen

#### Cel i sens analityczny

Chcemy przypisać każdą kwestię do sceny, w której pada — to pozwoli później śledzić emocje w czasie. Najpierw więc odnajdujemy granice scen, tak jak w module sieciowym.

#### Prompt dla modelu

```text
Kontekst:
W scenariuszach nowe sceny otwierają krótkie nagłówki opisujące miejsce i czas akcji, często zaczynające się od skrótów oznaczających wnętrze lub plener.

Wejście:
Oczyszczony tekst scenariusza z poprzedniego kroku.

Zadanie:
Wykryj granice scen i zbuduj uporządkowaną listę scen z numerami i nagłówkami. Zachowaj wynik tak, aby w kolejnym kroku dało się przypisać każdą kwestię do numeru sceny.

Pokaż wynik:
- łączną liczbę rozpoznanych scen,
- pierwszych 8 rekordów w formie `numer sceny | nagłówek sceny`.

Warunek poprawności:
Nagłówki powinny wyglądać jak opisy miejsca i czasu akcji, a nie jak dialog.

Jeśli wystąpi błąd:
Wyjaśnij, że nie udało się znaleźć wyraźnych granic scen albo że nagłówki są niejednoznaczne.

Nie rób jeszcze:
Nie przypisuj kwestii do scen ani postaci.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Liczbę scen rozpoznanych w całym tekście.
- Krótką listę pierwszych nagłówków scen.

### Krok 2B. Wydobycie kwestii dialogowych

#### Cel i sens analityczny

To rdzeń całego modułu. W zapisie scenariusza kwestia dialogowa ma stały wzór: **nazwa postaci** (wielkimi literami, wcięta), a pod nią **tekst wypowiedzi**. Wydobędziemy te pary i przypiszemy je do scen.

#### Prompt dla modelu

```text
Kontekst:
W scenariuszu kwestia dialogowa zwykle wygląda tak: w osobnej linii nazwa postaci zapisana wielkimi literami i wcięta, a bezpośrednio pod nią jedna lub kilka linii wypowiedzi. Nazwa może mieć dopisek w nawiasie (np. wskazówkę aktorską), który należy usunąć. Linie pisane od lewego marginesu to zwykle didaskalia, a nie wypowiedzi.

Wejście:
Oczyszczony tekst scenariusza oraz lista scen z poprzednich kroków.

Zadanie:
Przejdź przez tekst i wydobądź wszystkie kwestie dialogowe. Dla każdej kwestii zapisz: numer sceny, w której pada, oczyszczoną nazwę postaci, treść wypowiedzi oraz liczbę słów w tej wypowiedzi. Nazwy postaci oczyść tak samo jak w module sieciowym (wielkie litery, bez dopisków w nawiasach). Pomiń didaskalia, nagłówki scen i przejścia montażowe. Zachowaj wynik jako tabelę dostępną w kolejnych krokach.

Pokaż wynik:
- łączną liczbę wydobytych kwestii,
- liczbę unikalnych postaci mówiących,
- próbkę 15 wierszy w formie `numer sceny | postać | liczba słów | początek kwestii`.

Warunek poprawności:
W kolumnie postaci powinny dominować imiona lub nazwy postaci, a treść kwestii powinna wyglądać jak wypowiedź, a nie jak opis kamery czy nagłówek sceny. Liczba słów powinna zgadzać się z długością wypowiedzi.

Jeśli wystąpi błąd:
Pokaż kilka problematycznych fragmentów i wyjaśnij, czy problem dotyczy nietypowego formatowania nazw postaci, czy mylenia didaskaliów z wypowiedziami.

Nie rób jeszcze:
Nie licz jeszcze sumarycznej mowy postaci ani nie analizuj płci.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Łączną liczbę kwestii i liczbę unikalnych postaci mówiących.
- Próbkę wierszy: numer sceny, postać, liczba słów i początek wypowiedzi.
- Wypowiedzi, które realnie wyglądają jak dialog — bez didaskaliów i nagłówków.

---
## Etap 3: Kto mówi najwięcej?

Mamy już wszystkie kwestie. Teraz liczymy, ile mowy przypada na każdą postać — to pierwsza, prosta miara „głosu" w narracji.

### Krok 3A. Ranking postaci według ilości mowy

#### Cel i sens analityczny

Liczba wypowiedzianych słów pokazuje, czyimi ustami prowadzona jest opowieść. To inna miara niż liczba scen z modułu sieciowego — można być w wielu scenach i milczeć albo zdominować dialog w niewielu.

#### Prompt dla modelu

```text
Kontekst:
Masz tabelę kwestii dialogowych z przypisaną postacią i liczbą słów.

Wejście:
Tabela kwestii z poprzedniego kroku.

Zadanie:
Dla każdej postaci policz: łączną liczbę słów, liczbę kwestii oraz liczbę różnych scen, w których zabiera głos. Posortuj malejąco według liczby słów. Zachowaj wynik do kolejnych kroków.

Pokaż wynik:
- łączną liczbę postaci mówiących,
- tabelę 20 najwięcej mówiących postaci w formie `postać | liczba słów | liczba kwestii | liczba scen`.

Warunek poprawności:
Suma słów wszystkich postaci powinna odpowiadać łącznej liczbie słów we wszystkich kwestiach.

Jeśli wystąpi błąd:
Wyświetl informację, czy tabela kwestii jest pusta, czy liczby słów się nie sumują.

Nie rób jeszcze:
Nie rysuj wykresu i nie analizuj płci.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Tabelę rankingową najwięcej mówiących postaci.
- Dla każdej postaci trzy liczby: słowa, kwestie, sceny.
- Wyniki posortowane malejąco według liczby słów.

#### Pytanie interpretacyjne

Porównaj ten ranking z rankingiem liczby scen z modułu sieciowego. Czy najczęściej *obecne* postacie to te same, co najwięcej *mówiące*? Czy jest ktoś, kto „dużo znaczy w sieci, ale mało mówi" — albo odwrotnie?

---
## Etap 4: Płeć postaci

Żeby zapytać o reprezentację — kto w tym filmie ma głos — potrzebujemy przypisać postaciom płeć. Zrobi to model, a Ty zweryfikujesz wynik.

### Krok 4A. Otagowanie postaci płcią

#### Cel i sens analityczny

Płeć postaci to *atrybut*, który pozwoli spojrzeć na rozkład mowy pod kątem reprezentacji (np. w duchu testu Bechdel). To zarazem moment krytyczny: płeć przypisana automatycznie bywa uproszczeniem, dlatego wynik trzeba przejrzeć ręcznie.

**Podpowiedź:** ten krok różni się od poprzednich. Przekaż modelowi konkretną **listę postaci z rankingu z Kroku 3A** — model potrzebuje nazw, żeby przypisać płeć, korzystając z wiedzy o filmie.

#### Prompt dla modelu

```text
Kontekst:
Masz listę postaci mówiących z konkretnego filmu. Chcesz przypisać każdej z nich płeć, korzystając z wiedzy o filmie oraz z kontekstu wypowiedzi.

Wejście:
Lista postaci z rankingu mowy (wklejam ją poniżej promptu).

Zadanie:
Dla każdej postaci przypisz płeć jako jedną z wartości: "K" (kobieta), "M" (mężczyzna) albo "?" (niejednoznaczna, zbiorowa lub nieznana). Zwróć wynik jako czytelne, łatwe do ręcznej edycji przypisanie postać → płeć (np. zwykły słownik), tak aby dało się je poprawić jedną linijką. Zachowaj wynik do kolejnych kroków.

Pokaż wynik:
- przypisanie postać → płeć dla wszystkich postaci z listy,
- zbiorcze podsumowanie: ile postaci oznaczono jako K, M oraz ?.

Warunek poprawności:
Każda postać z listy powinna mieć dokładnie jedną z wartości K, M, ?. Postacie zbiorowe (np. tłum) i niejednoznaczne oznacz jako ?.

Jeśli wystąpi błąd:
Wskaż postacie, których płci nie da się rozstrzygnąć, i wyjaśnij dlaczego.

Nie rób jeszcze:
Nie rysuj wykresu.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Przypisanie płci do każdej postaci oraz podsumowanie liczby K / M / ?.
- Wynik w formie, którą da się ręcznie poprawić.

> **Sprawdź ręcznie.** Model się myli — zwłaszcza przy postaciach drugoplanowych, zbiorowych albo o niejednoznacznych imionach. Popraw oczywiste błędy, zanim przejdziesz dalej. To część analizy, nie obejście problemu: zastanów się, które przypadki są naprawdę trudne i dlaczego.

---
## Etap 5: Wykres — udział w dialogu według płci

Łączymy ranking mowy z płcią w jeden czytelny obraz. To pierwszy „wykres-bohater", który trafi później do prezentacji.

### Krok 5A. Wykres udziału w dialogu z podziałem na płeć

#### Cel i sens analityczny

Słupkowy wykres mowy, pokolorowany płcią, pokazuje od razu dwie rzeczy naraz: kto dominuje dialog i jak rozkłada się głos między kobiety a mężczyzn.

#### Prompt dla modelu

```text
Kontekst:
Masz ranking postaci według liczby słów oraz przypisaną każdej postaci płeć.

Wejście:
Ranking mowy i przypisanie płci.

Zadanie:
Narysuj poziomy wykres słupkowy dla 15 najwięcej mówiących postaci, gdzie długość słupka to liczba słów, a kolor oznacza płeć (K, M, ?). Dodaj tytuł, opis osi i legendę płci. Pod wykresem podaj jedną liczbę: jaki procent wszystkich wypowiedzianych słów przypada na postacie kobiece.

Pokaż wynik:
- jeden czytelny wykres z 15 postaciami, posortowany malejąco,
- legendę kolorów płci,
- procent słów wypowiedzianych przez kobiety.

Warunek poprawności:
Jeśli postaci jest mniej niż 15, narysuj tyle, ile jest. Kolory muszą zgadzać się z przypisaną płcią.

Jeśli wystąpi błąd:
Wyjaśnij, dlaczego nie da się przygotować wykresu (np. brak rankingu albo brak płci).

Nie rób jeszcze:
Nie zapisuj jeszcze plików.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Jeden poziomy wykres 15 postaci, kolorowany płcią, z tytułem i legendą.
- Pod wykresem procent słów wypowiedzianych przez postacie kobiece.

#### Pytanie interpretacyjne

Jaki procent dialogu należy do kobiet? Czy ten film „przeszedłby" intuicyjny test reprezentacji? Pamiętaj o granicy metody: udział w mowie to nie to samo co znaczenie postaci — milczenie też bywa znaczące.

---
## Etap 6: Eksport danych

Zapisujemy dwa pliki, które będą wejściem do następnego notatnika (analiza emocji) i do końcowej prezentacji.

### Krok 6A. Zapis plików `dialogi.csv` i `postacie.csv`

#### Cel i sens analityczny

Rozdzielamy dane na dwa poziomy: pojedyncze kwestie (`dialogi.csv`) i podsumowanie postaci wraz z płcią (`postacie.csv`). Taki podział wystarczy do analizy emocji w kolejnym module.

#### Prompt dla modelu

```text
Kontekst:
Masz tabelę kwestii, ranking mowy oraz przypisaną postaciom płeć.

Wejście:
Tabela kwestii i tabela postaci z płcią.

Zadanie:
Zapisz dwa pliki CSV. Plik `dialogi.csv` ma zawierać po jednym wierszu na kwestię, z kolumnami: numer sceny, postać, treść kwestii, liczba słów. Plik `postacie.csv` ma zawierać po jednym wierszu na postać, z kolumnami: postać, płeć, liczba słów, liczba kwestii, liczba scen. Zadbaj o poprawne kodowanie polskich znaków (UTF-8).

Pokaż wynik:
- komunikat o zapisaniu obu plików i liczbie wierszy w każdym,
- nazwy zapisanych plików,
- wskazówkę, jak pobrać je z Colab.

Warunek poprawności:
Oba pliki powinny być niepuste i otwierać się poza notatnikiem. Liczba wierszy w `dialogi.csv` powinna odpowiadać liczbie kwestii, a w `postacie.csv` liczbie postaci.

Jeśli wystąpi błąd:
Wyjaśnij, którego pliku nie udało się zapisać i dlaczego.

Nie rób już nic więcej.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Potwierdzenie zapisania plików `dialogi.csv` i `postacie.csv` wraz z liczbą wierszy.
- Wskazówkę, jak pobrać pliki z Google Colab.

---
## Co dalej?

- Masz teraz `dialogi.csv` (pojedyncze wypowiedzi) i `postacie.csv` (postacie z płcią i ilością mowy).
- W następnym notatniku ocenisz **emocje** każdej kwestii i narysujesz **łuk emocjonalny filmu** oraz rozkład **emocji według płci**.
- Na końcu połączysz te wykresy z siecią z pierwszego modułu w jedną prezentację.
- Jeśli któryś krok zadziałał źle, poproś model o poprawienie tylko tego jednego fragmentu, zamiast generować wszystko od nowa.